# Audit screening tests

Five screening tests on the cleaned GeBIZ data, for choosing what to look at more closely. None of them is evidence of irregularity, and nothing below identifies wrongdoing by any agency or supplier.

Unless stated otherwise the unit is an ETT tender (awards summed per `tender_no`), matching the threshold work in notebook 01. Methods and thresholds are in `../src/audit_tests.py`; the category keyword dictionary is in `../src/categories.py`.

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
from IPython.display import display

from analysis import aggregate_by_tender, top_suppliers
from audit_tests import (RISK_WEIGHTS, category_concentration, incumbency, risk_scores,
                         round_number_shares, tender_lite_comparison, with_category)

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

df = pd.read_csv('../data/processed/gebiz_cleaned.csv', low_memory=False)
ett_tenders = aggregate_by_tender(df).query("procurement_type == 'ETT'")
print(f"rows: {len(df):,} | ETT tenders: {len(ett_tenders):,}")

rows: 17,825 | ETT tenders: 10,909


## 1. Round numbers

Share of amounts that are exact multiples of S\$1,000 and S\$10,000, among amounts of at least S\$10. Round amounts are common in ordinary budgeting and there is no "correct" share to compare against, so the agency figures are read against the overall figure.

In [2]:
rounds = round_number_shares(df)
display(rounds.head(1))
display(round_number_shares(ett_tenders, group_col=None))
agencies = rounds.iloc[1:]
print(f"agencies with at least 300 amounts: {len(agencies)}")
print(f"multiples of 1,000: {agencies.share_x1000.min():.1%} to {agencies.share_x1000.max():.1%}")
print(f"multiples of 10,000: {agencies.share_x10000.min():.1%} to {agencies.share_x10000.max():.1%}")
print(f"agencies above the overall share: x1,000 {(agencies.share_x1000 > rounds.share_x1000.iloc[0]).sum()}, x10,000 {(agencies.share_x10000 > rounds.share_x10000.iloc[0]).sum()}")
display(agencies.head(5))

,group,n,n_x1000,share_x1000,n_x10000,share_x10000
0,all,16057,3104,0.193311,1140,0.070997


,group,n,n_x1000,share_x1000,n_x10000,share_x10000
0,all,10565,2279,0.215712,925,0.087553


agencies with at least 300 amounts: 18
multiples of 1,000: 5.5% to 33.5%
multiples of 10,000: 0.8% to 17.6%
agencies above the overall share: x1,000 8, x10,000 7


,group,n,n_x1000,share_x1000,n_x10000,share_x10000
1,Housing and Development Board,1173,393,0.335038,207,0.176471
2,Land Transport Authority,569,169,0.297012,93,0.163445
3,Ministry of Digital Development and Information,305,95,0.311475,36,0.118033
4,Ministry of Education,588,150,0.255102,60,0.102041
5,Public Utilities Board,708,184,0.259887,70,0.098870


**Reading.** 19.3% of row amounts and 21.6% of tender totals are multiples of S\$1,000; 7.1% and 8.8% are multiples of S\$10,000. Across the 18 agencies with at least 300 amounts, the S\$1,000 share runs from 5.5% to 33.5% and the S\$10,000 share from 0.8% to 17.6%; 8 and 7 agencies respectively sit above the overall share. Round amounts follow from budgets, rate schedules and fixed price lists, so a high share raises a question about how prices are set, not a problem in itself.

## 2. Long-running supply relationships

For each agency-supplier pair: the number of fiscal years with an award, and the pair's share of that agency's total awarded amount. A pair is flagged when it appears in at least 4 of the 5 fiscal years and holds at least 10% of the agency's spend.

Long relationships are normal for term contracts and specialised work. The flag marks relationships to understand, not suppliers to suspect. Supplier names are normalised for formatting only, so a supplier recorded under varying names is understated here.

In [3]:
pairs = incumbency(df)
print(f"agency-supplier pairs: {len(pairs):,}")
print('pairs by number of fiscal years:', pairs.years.value_counts().sort_index().to_dict())
print(f"flagged pairs: {int(pairs.flagged.sum())}, across {pairs[pairs.flagged].agency.nunique()} agencies")
display(pairs[pairs.flagged].round({'share_of_agency': 3}))

agency-supplier pairs: 11,890
pairs by number of fiscal years: {1: 8982, 2: 2020, 3: 622, 4: 181, 5: 85}
flagged pairs: 8, across 7 agencies


,agency,supplier_name,years,n_tenders,awarded_amt,share_of_agency,flagged
0,Ministry of Finance - Singapore Customs,NCS PTE LTD,4,5,1.921730e+08,0.731,True
1,Temasek Polytechnic,AVEPOINT SINGAPORE PTE LTD,4,5,5.667862e+07,0.288,True
2,National Environment Agency,800 SUPER WASTE MANAGEMENT PTE LTD,4,4,9.376931e+08,0.226,True
3,Central Provident Fund Board,NCS PTE LTD,5,15,4.690102e+07,0.221,True
4,Government Technology Agency (GovTech),NCS PTE LTD,5,10,5.531212e+07,0.206,True
5,Health Sciences Authority,WSH EXPERTS PTE LTD,4,7,6.019872e+07,0.187,True
6,Temasek Polytechnic,ASA CONTRACTS PTE LTD,4,9,3.559285e+07,0.181,True
7,Energy Market Authority of Singapore,ERNST & YOUNG ADVISORY PTE LTD,4,5,1.251038e+07,0.124,True


**Reading.** 8 pairs out of 11,890 are flagged, across 7 agencies, holding 12.4% to 73.1% of their agency's spend over 4 to 15 tenders. Most relationships are short: 8,982 pairs appear in a single fiscal year and only 266 in 4 or 5 years. Term contracts running several years produce exactly this pattern, so the flag says where to check how the work was re-tendered.

## 3. Categories

Keyword classification of `tender_description`. The first match wins, in the order consultancy, IT, construction, cleaning/facilities, training, supplies; anything unmatched is "other". Two rounds of manual spot-checks were done: 50 random tenders, then 30 tenders whose label changed after the keywords were extended.

In [4]:
tenders_cat = with_category(ett_tenders)
summary = tenders_cat.groupby('category').agg(tenders=('tender_no', 'size'), amount=('awarded_amt', 'sum'))
summary['share_tenders'] = summary.tenders / len(tenders_cat)
summary['share_amount'] = summary.amount / tenders_cat.awarded_amt.sum()
display(summary.drop(columns='amount').sort_values('tenders', ascending=False).round(3))
print(f"unclassified ('other'): {(tenders_cat.category == 'other').mean():.1%} of tenders")

,tenders,share_tenders,share_amount
category,,,
other,4077,0.374,0.212
supplies,1579,0.145,0.044
cleaning/facilities,1566,0.144,0.113
construction,1246,0.114,0.578
consultancy,1124,0.103,0.018
IT,1005,0.092,0.030
training,312,0.029,0.004


unclassified ('other'): 37.4% of tenders


**Reading.** 37.4% of tenders match no keyword and stay "other", holding 21.2% of the amount, so every category result below covers only part of the data. Construction is 11.4% of tenders but 57.8% of the amount, while consultancy, IT and training together are 22.4% of tenders and 5.2% of the amount. Known misses from the spot-checks: facility management at a data centre goes to IT, and one financial-system tender went to cleaning/facilities because its description says "maintenance".

## 4. Concentration by agency and category

Amount-weighted HHI and CR4 for each agency and category with at least 10 award rows, leaving out "other". Concentration inside a category says more than concentration at agency level, where unrelated purchases are mixed together.

In [5]:
conc = category_concentration(df)
print(f"groups: {len(conc)} across {conc.agency.nunique()} agencies")
print('groups by category:', conc.category.value_counts().to_dict())
print(f"HHI above 2,500: {int((conc.hhi > 2500).sum())}; above 5,000: {int((conc.hhi > 5000).sum())}")
display(conc.groupby('category').hhi.describe()[['count', '25%', '50%', '75%', 'max']].round(0))
display(conc.head(10).round({'hhi': 0, 'cr4': 1}))

groups: 215 across 69 agencies
groups by category: {'consultancy': 46, 'IT': 43, 'supplies': 43, 'cleaning/facilities': 39, 'construction': 27, 'training': 17}
HHI above 2,500: 87; above 5,000: 18


,count,25%,50%,75%,max
category,,,,,
IT,43.0,1892.0,2844.0,3650.0,9990.0
cleaning/facilities,39.0,1324.0,2193.0,3772.0,7548.0
construction,27.0,1380.0,2435.0,3877.0,9653.0
consultancy,46.0,1265.0,1899.0,2762.0,4920.0
supplies,43.0,1109.0,1730.0,2866.0,5029.0
training,17.0,1423.0,1652.0,2853.0,9105.0


,agency,category,hhi,n_suppliers,total,cr4,n_awards
0,Competition and Consumer Commission of Singapo...,IT,9990.0,10,3.270210e+05,100.0,10
1,Science Centre Board,construction,9653.0,13,5.865754e+08,99.6,21
2,Singapore Sports Council (Sport Singapore),training,9105.0,20,6.882010e+07,96.9,22
3,Ministry of Education,training,8813.0,27,8.929834e+07,97.1,35
4,Singapore Sports Council (Sport Singapore),construction,8153.0,16,1.132586e+09,98.8,18
5,Ministry of Finance - Singapore Customs,cleaning/facilities,7548.0,10,2.078486e+08,99.2,15
6,Singapore Land Authority,construction,6603.0,12,3.772350e+08,98.2,14
7,Science Centre Board,cleaning/facilities,6304.0,11,2.217904e+07,92.6,12
8,Land Transport Authority,IT,6173.0,11,1.174095e+08,98.2,20
9,Ministry of Health-Ministry Headquarter,cleaning/facilities,6156.0,10,3.253363e+08,97.2,11


**Reading.** 215 agency-category groups qualify, across 69 agencies; 87 are above HHI 2,500 and 18 above 5,000. Median HHI is highest for IT (2,844) and lowest for training (1,652). High concentration inside a narrow category is not surprising on its own: specialised systems and works often have few able suppliers, and a group can show 10 suppliers with CR4 at 100% when the top four hold all the value.

## 5. Tender Lite at S\$1 million

Tender Lite is a lighter tender process for procurements with an estimated value up to S\$1 million. It applies to general goods and services called from end April 2024 and to construction from May 2025; ICT only from April 2026, after this data ends, so IT serves as a control group that should show no change.

If the limit changed behaviour, tenders would bunch below S\$1 million after the start date. The data holds award dates only, so tenders awarded within six months after each start date are left out: they were probably called before it.

In [6]:
display(tender_lite_comparison(df).round({'share_below': 3, 'fisher_p': 3}))

,group,cut_date,period,n_below,n_above,share_below,fisher_p,excluded_buffer
0,goods/services,2024-05-01,before,92,58,0.613,NaN,NaN
1,goods/services,2024-05-01,after,53,26,0.671,0.238,27.0
2,construction,2025-05-01,before,17,21,0.447,NaN,NaN
3,construction,2025-05-01,after,2,0,1.000,0.219,5.0
4,IT (control),2024-05-01,before,10,8,0.556,NaN,NaN
5,IT (control),2024-05-01,after,7,2,0.778,0.244,5.0


**Reading.** No evidence of a Tender Lite effect. For goods and services, the share just below S\$1 million moves from 61.3% to 67.1% (Fisher p = 0.238). The IT control group, where the rule does not apply until after this data ends, moves the same way (55.6% to 77.8%, p = 0.244), which is what a common cause rather than the rule looks like. Construction has only 2 tenders after its start date, so it says nothing. The test is small, and award dates are a weak stand-in for when a tender was called.

## 6. Risk score and audit sample

Each ETT tender scores one weighted point per red flag. Weights are at the top of `audit_tests.py`; near-threshold carries half weight because notebook 01 found no clustering below S\$90,000 at tender level.

The top 50 tenders go to `../outputs/audit_sample.csv` with the reasons for each. This is a sample-selection aid, not a finding.

In [7]:
print('weights:', RISK_WEIGHTS)
scores = risk_scores(df)
print('score distribution:', scores.score.value_counts().sort_index(ascending=False).to_dict())
print('flag counts:', {flag: int(scores[flag].sum()) for flag in RISK_WEIGHTS})
print(f"tenders with two or more flags: {int((scores.score >= 2).sum())}")

sample = scores.head(50)
sample.to_csv('../outputs/audit_sample.csv', index=False)
print('reasons in the top 50:', sample.reasons.value_counts().to_dict())
display(sample[['tender_no', 'agency', 'category', 'fiscal_year', 'awarded_amt', 'score', 'reasons']].head(15))

weights: {'near_threshold': 0.5, 'round_amount': 1.0, 'high_share_incumbent': 1.0, 'dominant_supplier_category': 1.0}


score distribution: {2.0: 14, 1.0: 987, 0.5: 142, 0.0: 9766}
flag counts: {'near_threshold': 142, 'round_amount': 925, 'high_share_incumbent': 60, 'dominant_supplier_category': 30}
tenders with two or more flags: 14
reasons in the top 50: {'round_amount': 31, 'round_amount; high_share_incumbent': 8, 'high_share_incumbent; dominant_supplier_category': 4, 'dominant_supplier_category': 3, 'round_amount; dominant_supplier_category': 2, 'high_share_incumbent': 2}


,tender_no,agency,category,fiscal_year,awarded_amt,score,reasons
0,NHB000ETT22000015,National Heritage Board,construction,FY2023,2.356600e+08,2.0,round_amount; dominant_supplier_category
1,FINCEDETT21000007,Ministry of Finance - Singapore Customs,cleaning/facilities,FY2022,9.607138e+07,2.0,high_share_incumbent; dominant_supplier_category
2,MOE000ETT25000018,Ministry of Education,training,FY2025,8.380000e+07,2.0,round_amount; dominant_supplier_category
3,FINCEDETT21000004,Ministry of Finance - Singapore Customs,cleaning/facilities,FY2021,4.153917e+07,2.0,high_share_incumbent; dominant_supplier_category
4,FINCEDETT24000009,Ministry of Finance - Singapore Customs,cleaning/facilities,FY2025,3.725746e+07,2.0,high_share_incumbent; dominant_supplier_category
5,GVT000ETT21000017,Government Technology Agency (GovTech),cleaning/facilities,FY2021,3.150000e+07,2.0,round_amount; high_share_incumbent
6,TPO000ETT24000010,Temasek Polytechnic,other,FY2024,7.250000e+06,2.0,round_amount; high_share_incumbent
7,TPO000ETT22000007,Temasek Polytechnic,other,FY2022,5.370000e+06,2.0,round_amount; high_share_incumbent
8,TPO000ETT24000016,Temasek Polytechnic,other,FY2024,5.320000e+06,2.0,round_amount; high_share_incumbent
9,FINCEDETT21000006,Ministry of Finance - Singapore Customs,cleaning/facilities,FY2022,4.868427e+06,2.0,high_share_incumbent; dominant_supplier_category


**Reading.** The flags fire unevenly: 925 round amounts, 142 near-threshold, 60 high-share incumbents, 30 dominant-supplier. Only 14 tenders trip two flags and none trips three, so the top 50 is those 14 followed by 36 single-flag tenders ordered by amount. Read it as a starting list for sampling, not a ranking of risk. At half weight, near-threshold never lifts a tender into the top 50 on its own.

## 7. Check on Finding 4

The README says the ten largest suppliers by amount are all construction or engineering firms, which was read off their names. The categories allow a check: which categories do those suppliers' tenders fall into?

In [8]:
rows_cat = with_category(df)
top10 = top_suppliers(df, 10)
check = []
for name in top10.supplier_name:
    rows_s = rows_cat[rows_cat.supplier_name == name]
    by_amt = rows_s.groupby('category').awarded_amt.sum().sort_values(ascending=False)
    check.append({'supplier': name, 'n_rows': len(rows_s), 'total_amt': rows_s.awarded_amt.sum(),
                  'main_category': by_amt.index[0], 'main_share': by_amt.iloc[0] / by_amt.sum(),
                  'categories': ', '.join(by_amt.index)})
check = pd.DataFrame(check)
display(check.round({'main_share': 2}))
print('main category counts:', check.main_category.value_counts().to_dict())

,supplier,n_rows,total_amt,main_category,main_share,categories
0,RICH CONSTRUCTION COMPANY PTE LTD,9,3.539567e+09,construction,1.00,construction
1,KTC CIVIL ENGINEERING & CONSTRUCTION PTE LTD,19,2.803541e+09,construction,0.80,"construction, other, cleaning/facilities"
2,WOH HUP PRIVATE LIMITED,6,2.508696e+09,construction,0.87,"construction, other"
3,OBAYASHI SINGAPORE PTE LTD,2,2.391261e+09,other,0.76,"other, construction"
4,TEAMBUILD ENGINEERING & CONSTRUCTION PTE LTD,9,1.844377e+09,construction,1.00,"construction, other"
5,PENTA-OCEAN CONSTRUCTION COMPANY LIMITED,4,1.828634e+09,construction,0.84,"construction, other"
6,NEWCON BUILDERS PTE LTD,9,1.789420e+09,construction,1.00,construction
7,KAJIMA OVERSEAS ASIA SINGAPORE PTE LTD,2,1.735619e+09,construction,1.00,construction
8,BHCC CONSTRUCTION PTE LTD,6,1.567928e+09,construction,0.89,"construction, IT"
9,SHANGHAI TUNNEL ENGINEERING CO SINGAPORE PTE LTD,3,1.426000e+09,construction,1.00,construction


main category counts: {'construction': 9, 'other': 1}


**Reading.** 9 of the 10 largest suppliers have construction as their main category by amount. The tenth has 76% of its amount in "other" because its two tender descriptions match no construction keyword, which is a gap in the dictionary rather than a different line of business. The README claim holds, but it rests partly on reading company names.